# California General Land Use Data Gather

California General Land Use dataset has 534,346 rows; however, esri api export limits the row count to 2000. The api delivered data comes with geometry details of the data which will be used to map the lcoations against MSSA data and get the FIPS code for each location. loop

In [0]:
column_rename_dict = {'attributes.OBJECTID': 'OBJECTID', 
                      'attributes.County': 'County', 
                      'attributes.jurisdiction': 'jurisdiction', 
                      'attributes.classkey': 'classkey', 
                      'attributes.code': 'code', 
                      'attributes.description': 'description', 
                      'attributes.ucd_number': 'ucd_number', 
                      'attributes.ucd_description': 'ucd_description', 
                      'attributes.Source': 'Source', 
                      'attributes.Date': 'Date', 
                      'attributes.Shape__Area': 'Shape__Area', 
                      'attributes.Shape__Length': 'Shape__Length', 
                      'geometry.rings': 'geometry_rings'}


As the python kernel shuts down to store large number of rows in memory, the data needs to be stored in Deltalake as they are coming. The code is set to process every 50,000 rows of data at a time which is equivalent to 25 batches of 2000 records 

In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession


base_url = "https://services8.arcgis.com/Xr1lDrwMv89PhjD9/arcgis/rest/services/California_General_Plan_Land_Use/FeatureServer/2/query"

spark = SparkSession.builder.getOrCreate()

offset = 0
batch_count = 0
N_batches = 25              # to process 50,000 records at a time
total_data_row_count = 534346
record_holder = []
first_batch = True

while offset < total_data_row_count:
    params = {
    "where": "1=1",
    "outFields": "*",
    "outSR": "4326",
    "f": "json",
    "resultOffset" : offset,
    "resultRecordCount": 2000
    }
    
    ca_land_response = requests.get(base_url, params=params)

    if ca_land_response.status_code != 200:
        print(f"Error: {ca_land_response.status_code}")
        break
    
    ca_land_data = ca_land_response.json()
    features = ca_land_data.get('features', [])
    
    record_holder.extend(features)
    offset += len(features)
    batch_count += 1

    last_batch = len(features) < 2000 or not ca_land_data.get("exceededTransferLimit", False)

    if batch_count % N_batches == 0 or last_batch:
        batch_pddf = pd.json_normalize(record_holder)
        batch_pddf.rename(columns=column_rename_dict, inplace=True)
        record_holder = []
        
        spark_df = spark.createDataFrame(batch_pddf)
        write_mode = "overwrite" if first_batch else "append"

        if first_batch:
            schema = spark_df.schema
            first_batch = False

        spark_df.write.mode(write_mode).saveAsTable("ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df")
        
        print(f"processed {batch_count} batches and {offset} records")



**The drop table code below is for the development stage**

In [0]:
delete_table = "DROP TABLE IF EXISTS ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df"
spark.sql(delete_table)